In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1791574678.807677     960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791574678.808068     960 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791574678.839925     960 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791574679.466099     960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
TICKER   = "AAPL"
PERIOD   = "10y"        
SEQ_LEN  = 60          
TEST_SPLIT = 0.2       
EPOCHS   = 50
BATCH    = 32

In [3]:
def download_data(ticker: str, period: str) -> pd.DataFrame:
    df = yf.download(ticker, period=period, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):   # handle newer yfinance multi-index
        df.columns = df.columns.get_level_values(0)
    return df


In [ ]:
def add_features(df):
    df = df.copy()
    df["ret_1"]  = df["Close"].pct_change()
    df["ma_5"]   = df["Close"].rolling(5).mean()
    df["ma_20"]  = df["Close"].rolling(20).mean()
    df["volatility"] = df["ret_1"].rolling(10).std()

    delta = df["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df["rsi"] = 100 - 100 / (1 + gain / loss)

    df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

    df = df.dropna()
    return df

In [ ]:
def build_sequences(data, targets, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(targets[i + seq_len])
    return np.array(X), np.array(y)

In [3]:
def main():
    df = download_data(TICKER, PERIOD)
    df = add_features(df)

    features = ["Open", "High", "Low", "Close", "Volume",
                "ma_5", "ma_20", "volatility", "rsi"]
    X_raw = df[features].values
    y_raw = df["target"].values

    split_idx = int(len(X_raw) * (1 - TEST_SPLIT))
    scaler = MinMaxScaler()
    scaler.fit(X_raw[:split_idx])
    X_scaled = scaler.transform(X_raw)


In [2]:
if __name__ == "__main__":
    main()

NameError: name 'download_data' is not defined